In [1]:
# CELL 0 - CLEAR OLD SPARK CACHE
# Safe for every full pipeline run

spark.catalog.clearCache()

print("SUCCESS: Old Spark cache cleared.")

StatementMeta(, 816ca5cb-8499-4444-8822-20e729a2e8b1, 3, Finished, Available, Finished, False)

SUCCESS: Old Spark cache cleared.


In [2]:
# CELL 1 - READ BRONZE CLINICAL ENCOUNTERS
# Rerunnable: reads the current Bronze Delta table

bronze_table_name = "bronze_clinical_encounters"

if not spark.catalog.tableExists(bronze_table_name):
    raise ValueError(
        f"Bronze table does not exist: {bronze_table_name}"
    )

bronze_clinical_encounters_df = spark.table(
    bronze_table_name
)

bronze_count = bronze_clinical_encounters_df.count()

if bronze_count == 0:
    raise ValueError(
        "Bronze clinical encounters table contains no rows."
    )

required_metadata_columns = [
    "batch_id",
    "source_file_name",
    "ingestion_timestamp"
]

missing_metadata_columns = [
    column_name
    for column_name in required_metadata_columns
    if column_name not in bronze_clinical_encounters_df.columns
]

if missing_metadata_columns:
    raise ValueError(
        f"Missing required Bronze metadata columns: "
        f"{missing_metadata_columns}"
    )

print("Bronze table:", bronze_table_name)
print("Bronze rows:", bronze_count)

print("Bronze columns:")
print(bronze_clinical_encounters_df.columns)

bronze_clinical_encounters_df.printSchema()

display(
    bronze_clinical_encounters_df.limit(20)
)

StatementMeta(, 816ca5cb-8499-4444-8822-20e729a2e8b1, 4, Finished, Available, Finished, False)

Bronze table: bronze_clinical_encounters
Bronze rows: 973589
Bronze columns:
['facility_city', 'attending_staff_id', 'billing_amount', 'encounter_id', 'encounter_timestamp', 'encountertype', 'facility_c', 'facility_id', 'insurance_claim_status', 'patient_id', 'primary_diagnosis_code', 'procedure_', 'provider_id', 'unexpected_source_field', 'batch_id', 'source_file_name', 'ingestion_timestamp']
root
 |-- facility_city: string (nullable = true)
 |-- attending_staff_id: string (nullable = true)
 |-- billing_amount: string (nullable = true)
 |-- encounter_id: string (nullable = true)
 |-- encounter_timestamp: string (nullable = true)
 |-- encountertype: string (nullable = true)
 |-- facility_c: string (nullable = true)
 |-- facility_id: string (nullable = true)
 |-- insurance_claim_status: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- primary_diagnosis_code: string (nullable = true)
 |-- procedure_: string (nullable = true)
 |-- provider_id: string (nullable = tru

SynapseWidget(Synapse.DataFrame, 206ac16a-3330-41c4-9669-cfe315ba1869)

In [3]:
# CELL 2 - STANDARDIZE SOURCE COLUMN NAMES AND TRIM STRINGS
# Rerunnable: always rebuilds from Bronze without writing any tables

from pyspark.sql.functions import col, trim
from pyspark.sql.types import StringType


# Map malformed Bronze column names to the required Silver names
column_rename_map = {
    "encountertype": "encounter_type",
    "facility_c": "facility_country",
    "procedure_": "procedure_code"
}


clinical_encounters_standardized_df = (
    bronze_clinical_encounters_df
)


# Rename only when the old column exists
for old_column, new_column in column_rename_map.items():

    if (
        old_column
        in clinical_encounters_standardized_df.columns
    ):

        clinical_encounters_standardized_df = (
            clinical_encounters_standardized_df
            .withColumnRenamed(
                old_column,
                new_column
            )
        )


# Required columns expected after renaming
required_business_columns = [
    "encounter_id",
    "patient_id",
    "provider_id",
    "facility_id",
    "encounter_timestamp",
    "encounter_type",
    "primary_diagnosis_code",
    "procedure_code",
    "billing_amount",
    "insurance_claim_status",
    "facility_city",
    "facility_country",
    "attending_staff_id"
]


missing_business_columns = [
    column_name
    for column_name in required_business_columns
    if column_name
    not in clinical_encounters_standardized_df.columns
]


if missing_business_columns:

    raise ValueError(
        f"Missing required clinical encounter columns: "
        f"{missing_business_columns}"
    )


# Trim leading and trailing spaces from all string columns
for field in clinical_encounters_standardized_df.schema.fields:

    if isinstance(field.dataType, StringType):

        clinical_encounters_standardized_df = (
            clinical_encounters_standardized_df
            .withColumn(
                field.name,
                trim(col(field.name))
            )
        )


standardized_count = (
    clinical_encounters_standardized_df.count()
)


if standardized_count != bronze_count:

    raise ValueError(
        f"Cell 2 row-count validation failed. "
        f"Bronze={bronze_count}, "
        f"Standardized={standardized_count}"
    )


print("Standardized rows:", standardized_count)

print("Standardized columns:")
print(clinical_encounters_standardized_df.columns)

print(
    "SUCCESS: Column names and string whitespace "
    "were standardized without changing row count."
)

clinical_encounters_standardized_df.printSchema()

display(
    clinical_encounters_standardized_df.limit(20)
)

StatementMeta(, 816ca5cb-8499-4444-8822-20e729a2e8b1, 5, Finished, Available, Finished, False)

Standardized rows: 973589
Standardized columns:
['facility_city', 'attending_staff_id', 'billing_amount', 'encounter_id', 'encounter_timestamp', 'encounter_type', 'facility_country', 'facility_id', 'insurance_claim_status', 'patient_id', 'primary_diagnosis_code', 'procedure_code', 'provider_id', 'unexpected_source_field', 'batch_id', 'source_file_name', 'ingestion_timestamp']
SUCCESS: Column names and string whitespace were standardized without changing row count.
root
 |-- facility_city: string (nullable = true)
 |-- attending_staff_id: string (nullable = true)
 |-- billing_amount: string (nullable = true)
 |-- encounter_id: string (nullable = true)
 |-- encounter_timestamp: string (nullable = true)
 |-- encounter_type: string (nullable = true)
 |-- facility_country: string (nullable = true)
 |-- facility_id: string (nullable = true)
 |-- insurance_claim_status: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- primary_diagnosis_code: string (nullable = true)
 |-

SynapseWidget(Synapse.DataFrame, 974a14f3-69af-4953-ae62-9f06bb760161)

In [4]:
# CELL 3 - CLEAN VALUES, STANDARDIZE FORMATS, AND CONVERT TYPES
# Rerunnable: rebuilds entirely from the Cell 2 dataframe

from pyspark.sql.functions import (
    col,
    when,
    lower,
    trim,
    upper,
    initcap,
    regexp_replace,
    expr
)
from pyspark.sql.types import StringType


null_like_values = [
    "",
    "null",
    "n/a",
    "na",
    "unknown",
    "undefined",
    "no value",
    "missing_id",
    "???",
    "??"
]


clinical_encounters_cleaned_df = (
    clinical_encounters_standardized_df
)


# Convert source null-like strings into actual NULL values
for field in clinical_encounters_cleaned_df.schema.fields:

    if isinstance(field.dataType, StringType):

        clinical_encounters_cleaned_df = (
            clinical_encounters_cleaned_df
            .withColumn(
                field.name,
                when(
                    lower(trim(col(field.name))).isin(
                        null_like_values
                    ),
                    None
                )
                .otherwise(trim(col(field.name)))
            )
        )


# Standardize identifiers, categories, codes and locations
clinical_encounters_cleaned_df = (
    clinical_encounters_cleaned_df

    .withColumn(
        "encounter_id",
        upper(col("encounter_id"))
    )

    .withColumn(
        "patient_id",
        upper(col("patient_id"))
    )

    .withColumn(
        "provider_id",
        upper(col("provider_id"))
    )

    .withColumn(
        "facility_id",
        upper(col("facility_id"))
    )

    .withColumn(
        "attending_staff_id",
        upper(col("attending_staff_id"))
    )

    .withColumn(
        "encounter_type",
        upper(col("encounter_type"))
    )

    .withColumn(
        "insurance_claim_status",
        upper(col("insurance_claim_status"))
    )

    .withColumn(
        "facility_country",
        upper(col("facility_country"))
    )

    .withColumn(
        "primary_diagnosis_code",
        upper(col("primary_diagnosis_code"))
    )

    .withColumn(
        "procedure_code",
        upper(col("procedure_code"))
    )

    .withColumn(
        "facility_city",
        initcap(lower(col("facility_city")))
    )
)


# Clean billing text before safe decimal conversion
clinical_encounters_cleaned_df = (
    clinical_encounters_cleaned_df

    .withColumn(
        "_billing_amount_clean",
        regexp_replace(
            col("billing_amount").cast("string"),
            r"[^0-9.\-]",
            ""
        )
    )

    .withColumn(
        "_billing_amount_clean",
        when(
            trim(col("_billing_amount_clean")) == "",
            None
        )
        .otherwise(col("_billing_amount_clean"))
    )

    .withColumn(
        "billing_amount",
        expr(
            "try_cast(_billing_amount_clean AS DECIMAL(18,2))"
        )
    )

    .drop("_billing_amount_clean")
)


# Safely convert timestamps
# Invalid dates become NULL and will be quarantined later
clinical_encounters_cleaned_df = (
    clinical_encounters_cleaned_df
    .withColumn(
        "encounter_timestamp",
        expr(
            "try_cast(encounter_timestamp AS TIMESTAMP)"
        )
    )
)


cleaned_count = clinical_encounters_cleaned_df.count()

if cleaned_count != bronze_count:

    raise ValueError(
        f"Cell 3 row-count validation failed. "
        f"Bronze={bronze_count}, "
        f"Cleaned={cleaned_count}"
    )


print("Cleaned rows:", cleaned_count)

print(
    "SUCCESS: Values, identifiers, categories, dates "
    "and billing amounts were standardized safely."
)

clinical_encounters_cleaned_df.printSchema()

display(
    clinical_encounters_cleaned_df.select(
        "encounter_id",
        "patient_id",
        "provider_id",
        "facility_id",
        "encounter_timestamp",
        "encounter_type",
        "primary_diagnosis_code",
        "procedure_code",
        "billing_amount",
        "insurance_claim_status",
        "facility_city",
        "facility_country",
        "attending_staff_id",
        "batch_id",
        "source_file_name",
        "ingestion_timestamp"
    ).limit(20)
)

StatementMeta(, 816ca5cb-8499-4444-8822-20e729a2e8b1, 6, Finished, Available, Finished, False)

Cleaned rows: 973589
SUCCESS: Values, identifiers, categories, dates and billing amounts were standardized safely.
root
 |-- facility_city: string (nullable = true)
 |-- attending_staff_id: string (nullable = true)
 |-- billing_amount: decimal(18,2) (nullable = true)
 |-- encounter_id: string (nullable = true)
 |-- encounter_timestamp: timestamp (nullable = true)
 |-- encounter_type: string (nullable = true)
 |-- facility_country: string (nullable = true)
 |-- facility_id: string (nullable = true)
 |-- insurance_claim_status: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- primary_diagnosis_code: string (nullable = true)
 |-- procedure_code: string (nullable = true)
 |-- provider_id: string (nullable = true)
 |-- unexpected_source_field: string (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- source_file_name: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)



SynapseWidget(Synapse.DataFrame, 01a8453f-6129-4cf4-90cf-3b6624915d68)

In [5]:
# CELL 4 - VALIDATE CLINICAL ENCOUNTER DATA
# Rerunnable: classifies every cleaned Bronze row as valid or rejected

from pyspark.sql.functions import (
    col,
    when,
    lit,
    concat_ws,
    current_timestamp,
    to_timestamp
)

valid_encounter_types = [
    "INPATIENT",
    "OUTPATIENT",
    "EMERGENCY"
]

valid_claim_statuses = [
    "APPROVED",
    "PENDING",
    "PAID",
    "DENIED"
]

minimum_valid_encounter_timestamp = to_timestamp(
    lit("2000-01-01 00:00:00")
)

clinical_encounters_validated_df = (
    clinical_encounters_cleaned_df
    .withColumn(
        "error_reason",
        concat_ws(
            "; ",

            when(
                col("encounter_id").isNull()
                | ~col("encounter_id").rlike(
                    r"^ENC_[0-9]{7}$"
                ),
                lit("Invalid or missing encounter_id")
            ),

            when(
                col("patient_id").isNull()
                | ~col("patient_id").rlike(
                    r"^PAT_[0-9]{6}$"
                ),
                lit("Invalid or missing patient_id")
            ),

            when(
                col("provider_id").isNull()
                | ~col("provider_id").rlike(
                    r"^PROV_[0-9]{4}$"
                ),
                lit("Invalid or missing provider_id")
            ),

            when(
                col("facility_id").isNull()
                | ~col("facility_id").rlike(
                    r"^FAC_[0-9]+$"
                ),
                lit("Invalid or missing facility_id")
            ),

            when(
                col("encounter_timestamp").isNull(),
                lit("Missing or invalid encounter_timestamp")
            ),

            when(
                col("encounter_timestamp").isNotNull()
                & (
                    col("encounter_timestamp")
                    < minimum_valid_encounter_timestamp
                ),
                lit("Encounter timestamp is before 2000-01-01")
            ),

            when(
                col("encounter_timestamp").isNotNull()
                & (
                    col("encounter_timestamp")
                    > current_timestamp()
                ),
                lit("Encounter timestamp is in the future")
            ),

            when(
                col("encounter_type").isNull()
                | ~col("encounter_type").isin(
                    valid_encounter_types
                ),
                lit("Invalid or missing encounter_type")
            ),

            when(
                col("primary_diagnosis_code").isNull()
                | ~col("primary_diagnosis_code").rlike(
                    r"^[A-Z][0-9]{2}(\.[0-9A-Z]+)?$"
                ),
                lit(
                    "Invalid or missing "
                    "primary_diagnosis_code"
                )
            ),

            when(
                col("procedure_code").isNull()
                | ~col("procedure_code").rlike(
                    r"^[0-9]{5}$"
                ),
                lit("Invalid or missing procedure_code")
            ),

            when(
                col("billing_amount").isNull()
                | (col("billing_amount") < 0)
                | (col("billing_amount") > 1000000),
                lit("Invalid billing_amount")
            ),

            when(
                col("insurance_claim_status").isNull()
                | ~col("insurance_claim_status").isin(
                    valid_claim_statuses
                ),
                lit(
                    "Invalid or missing "
                    "insurance_claim_status"
                )
            ),

            when(
                col("facility_city").isNull()
                | ~col("facility_city").rlike(
                    r"^[A-Za-z][A-Za-z .'-]*$"
                ),
                lit("Invalid or missing facility_city")
            ),

            when(
                col("facility_country").isNull()
                | (col("facility_country") != "CANADA"),
                lit("Invalid or missing facility_country")
            ),

            when(
                col("attending_staff_id").isNull()
                | ~col("attending_staff_id").rlike(
                    r"^EMP_[0-9]{4}$"
                ),
                lit(
                    "Invalid or missing "
                    "attending_staff_id"
                )
            )
        )
    )
)

valid_clinical_encounters_df = (
    clinical_encounters_validated_df
    .filter(col("error_reason") == "")
    .drop("error_reason")
)

quarantine_clinical_encounters_df = (
    clinical_encounters_validated_df
    .filter(col("error_reason") != "")
)

validated_count = clinical_encounters_validated_df.count()
valid_count = valid_clinical_encounters_df.count()
rejected_count = quarantine_clinical_encounters_df.count()

print("Validated rows:", validated_count)
print("Valid rows:", valid_count)
print("Rejected rows:", rejected_count)

if validated_count != valid_count + rejected_count:
    raise ValueError(
        f"Cell 4 reconciliation failed. "
        f"Validated={validated_count}, "
        f"Valid+Rejected={valid_count + rejected_count}"
    )

if validated_count != bronze_count:
    raise ValueError(
        f"Cell 4 source count failed. "
        f"Bronze={bronze_count}, "
        f"Validated={validated_count}"
    )

print(
    "SUCCESS: Every clinical encounter was classified "
    "as valid or rejected."
)

display(
    quarantine_clinical_encounters_df.select(
        "encounter_id",
        "patient_id",
        "provider_id",
        "facility_id",
        "encounter_timestamp",
        "encounter_type",
        "primary_diagnosis_code",
        "procedure_code",
        "billing_amount",
        "insurance_claim_status",
        "facility_city",
        "facility_country",
        "attending_staff_id",
        "error_reason"
    ).limit(20)
)

StatementMeta(, 816ca5cb-8499-4444-8822-20e729a2e8b1, 7, Finished, Available, Finished, False)

Validated rows: 973589
Valid rows: 1343
Rejected rows: 972246
SUCCESS: Every clinical encounter was classified as valid or rejected.


SynapseWidget(Synapse.DataFrame, 3bcb98a7-03ad-44cc-88e8-0917ec6aa2c7)

In [6]:
# CELL 5 - DEDUPLICATE VALID RECORDS AND FINALIZE QUARANTINE
# Rerunnable: keeps one deterministic row per encounter_id
# and sends additional duplicate rows to quarantine

from pyspark.sql import Window
from pyspark.sql.functions import (
    col,
    row_number,
    current_timestamp,
    lit,
    sha2,
    to_json,
    struct
)


# Create a deterministic tie-breaker from the complete valid row
deduplication_columns = sorted(
    valid_clinical_encounters_df.columns
)

valid_ranked_df = (
    valid_clinical_encounters_df

    .withColumn(
        "_deduplication_hash",
        sha2(
            to_json(
                struct(
                    *[
                        col(column_name)
                        for column_name
                        in deduplication_columns
                    ]
                )
            ),
            256
        )
    )
)


# Keep the newest record for each encounter_id
deduplication_window = (
    Window
    .partitionBy("encounter_id")
    .orderBy(
        col("ingestion_timestamp").desc_nulls_last(),
        col("encounter_timestamp").desc_nulls_last(),
        col("_deduplication_hash").desc()
    )
)


valid_ranked_df = (
    valid_ranked_df
    .withColumn(
        "_duplicate_rank",
        row_number().over(deduplication_window)
    )
)


# Final valid Silver candidates
deduplicated_clinical_encounters_df = (
    valid_ranked_df
    .filter(col("_duplicate_rank") == 1)
    .drop(
        "_duplicate_rank",
        "_deduplication_hash"
    )
)


# Additional rows with the same encounter_id go to quarantine
duplicate_clinical_encounters_df = (
    valid_ranked_df
    .filter(col("_duplicate_rank") > 1)

    .drop(
        "_duplicate_rank",
        "_deduplication_hash"
    )

    .withColumn(
        "error_reason",
        lit("Duplicate encounter_id")
    )

    .withColumn(
        "quarantine_timestamp",
        current_timestamp()
    )
)


# Add quarantine timestamp to data-quality rejections
data_quality_quarantine_df = (
    quarantine_clinical_encounters_df
    .withColumn(
        "quarantine_timestamp",
        current_timestamp()
    )
)


# Final quarantine includes:
# 1. Data-quality failures
# 2. Duplicate valid records not selected for Silver
final_quarantine_clinical_encounters_df = (
    data_quality_quarantine_df
    .unionByName(
        duplicate_clinical_encounters_df,
        allowMissingColumns=True
    )
)


valid_before_dedup_count = (
    valid_clinical_encounters_df.count()
)

deduplicated_count = (
    deduplicated_clinical_encounters_df.count()
)

duplicate_count = (
    duplicate_clinical_encounters_df.count()
)

quality_rejected_count = (
    quarantine_clinical_encounters_df.count()
)

final_quarantine_count = (
    final_quarantine_clinical_encounters_df.count()
)

final_reconciled_count = (
    deduplicated_count
    + final_quarantine_count
)


print("Valid rows before deduplication:", valid_before_dedup_count)
print("Silver candidates after deduplication:", deduplicated_count)
print("Duplicate rows quarantined:", duplicate_count)
print("Data-quality rows quarantined:", quality_rejected_count)
print("Final quarantine rows:", final_quarantine_count)
print("Final reconciled rows:", final_reconciled_count)


if valid_before_dedup_count != (
    deduplicated_count + duplicate_count
):
    raise ValueError(
        f"Deduplication reconciliation failed. "
        f"Valid before dedup={valid_before_dedup_count}, "
        f"Kept+duplicates="
        f"{deduplicated_count + duplicate_count}"
    )


if final_quarantine_count != (
    quality_rejected_count + duplicate_count
):
    raise ValueError(
        f"Quarantine reconciliation failed. "
        f"Final quarantine={final_quarantine_count}, "
        f"Quality rejected+duplicates="
        f"{quality_rejected_count + duplicate_count}"
    )


if bronze_count != final_reconciled_count:
    raise ValueError(
        f"Final source reconciliation failed. "
        f"Bronze={bronze_count}, "
        f"Silver candidates+quarantine="
        f"{final_reconciled_count}"
    )


duplicate_encounter_ids = (
    deduplicated_clinical_encounters_df
    .groupBy("encounter_id")
    .count()
    .filter(col("count") > 1)
    .count()
)


if duplicate_encounter_ids > 0:
    raise ValueError(
        f"Deduplication failed. "
        f"{duplicate_encounter_ids} encounter IDs "
        f"still appear more than once."
    )


print(
    "SUCCESS: Valid clinical encounters were deduplicated "
    "and every excluded duplicate was quarantined."
)


display(
    final_quarantine_clinical_encounters_df.select(
        "encounter_id",
        "patient_id",
        "facility_id",
        "batch_id",
        "source_file_name",
        "error_reason",
        "quarantine_timestamp"
    ).limit(20)
)

StatementMeta(, 816ca5cb-8499-4444-8822-20e729a2e8b1, 8, Finished, Available, Finished, False)

Valid rows before deduplication: 1343
Silver candidates after deduplication: 1290
Duplicate rows quarantined: 53
Data-quality rows quarantined: 972246
Final quarantine rows: 972299
Final reconciled rows: 973589
SUCCESS: Valid clinical encounters were deduplicated and every excluded duplicate was quarantined.


SynapseWidget(Synapse.DataFrame, 173cc4da-1957-4d8c-893c-69f33e08b50a)

In [7]:
# CELL 6 - REBUILD SILVER CLINICAL ENCOUNTERS
# Fully rerunnable: replaces Silver with the current clean,
# validated and deduplicated Bronze result on every pipeline run

from pyspark.sql.functions import (
    col,
    current_timestamp,
    sha2,
    concat_ws,
    coalesce,
    lit
)

silver_table_name = "silver_clinical_encounters"


approved_silver_columns = [
    "encounter_id",
    "patient_id",
    "provider_id",
    "facility_id",
    "encounter_timestamp",
    "encounter_type",
    "primary_diagnosis_code",
    "procedure_code",
    "billing_amount",
    "insurance_claim_status",
    "facility_city",
    "facility_country",
    "attending_staff_id",
    "batch_id",
    "source_file_name",
    "ingestion_timestamp"
]


hash_columns = [
    "encounter_id",
    "patient_id",
    "provider_id",
    "facility_id",
    "encounter_timestamp",
    "encounter_type",
    "primary_diagnosis_code",
    "procedure_code",
    "billing_amount",
    "insurance_claim_status",
    "facility_city",
    "facility_country",
    "attending_staff_id"
]


missing_silver_columns = [
    column_name
    for column_name in approved_silver_columns
    if column_name
    not in deduplicated_clinical_encounters_df.columns
]


if missing_silver_columns:

    raise ValueError(
        f"Missing required Silver columns: "
        f"{missing_silver_columns}"
    )


silver_clinical_source_df = (
    deduplicated_clinical_encounters_df

    .select(*approved_silver_columns)

    .withColumn(
        "row_hash",
        sha2(
            concat_ws(
                "||",
                *[
                    coalesce(
                        col(column_name).cast("string"),
                        lit("")
                    )
                    for column_name in hash_columns
                ]
            ),
            256
        )
    )

    .withColumn(
        "silver_processing_timestamp",
        current_timestamp()
    )
)


silver_source_count = (
    silver_clinical_source_df.count()
)


duplicate_silver_ids = (
    silver_clinical_source_df
    .groupBy("encounter_id")
    .count()
    .filter(col("count") > 1)
    .count()
)


if duplicate_silver_ids > 0:

    raise ValueError(
        f"Silver write stopped. "
        f"{duplicate_silver_ids} duplicate encounter IDs "
        f"remain in the source."
    )


# Overwrite removes stale or previously dirty Silver rows
(
    silver_clinical_source_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_table_name)
)


silver_target_count = (
    spark.table(silver_table_name).count()
)


print("Silver source rows:", silver_source_count)
print("Silver target rows:", silver_target_count)


if silver_source_count != silver_target_count:

    raise ValueError(
        f"Silver write verification failed. "
        f"Source={silver_source_count}, "
        f"Target={silver_target_count}"
    )


missing_silver_rows = (
    silver_clinical_source_df
    .select(
        "encounter_id",
        "row_hash"
    )
    .join(
        spark.table(silver_table_name)
        .select(
            "encounter_id",
            "row_hash"
        ),
        on=[
            "encounter_id",
            "row_hash"
        ],
        how="left_anti"
    )
    .count()
)


if missing_silver_rows > 0:

    raise ValueError(
        f"Silver verification failed. "
        f"{missing_silver_rows} records are missing "
        f"or have an incorrect row_hash."
    )


if "unexpected_source_field" in spark.table(
    silver_table_name
).columns:

    raise ValueError(
        "unexpected_source_field still exists in Silver."
    )


print(
    "SUCCESS: Silver clinical encounters was rebuilt "
    "without duplicates, stale dirty rows or unexpected columns."
)


display(
    spark.table(silver_table_name)
    .orderBy("encounter_id")
    .limit(20)
)

StatementMeta(, 816ca5cb-8499-4444-8822-20e729a2e8b1, 9, Finished, Available, Finished, False)

Silver source rows: 1290
Silver target rows: 1290
SUCCESS: Silver clinical encounters was rebuilt without duplicates, stale dirty rows or unexpected columns.


SynapseWidget(Synapse.DataFrame, 66057cea-0466-480c-b52d-952d28391949)

In [8]:
# CELL 7 - REBUILD QUARANTINE CLINICAL ENCOUNTERS
# Fully rerunnable and preserves every rejected Bronze row

from pyspark.sql import Window
from pyspark.sql.functions import (
    col,
    sha2,
    concat_ws,
    coalesce,
    lit,
    row_number
)

quarantine_table_name = "quarantine_clinical_encounters"


# Create a deterministic base hash for each rejected record
quarantine_with_base_hash_df = (
    final_quarantine_clinical_encounters_df

    .withColumn(
        "_quarantine_base_hash",
        sha2(
            concat_ws(
                "||",
                coalesce(col("batch_id").cast("string"), lit("")),
                coalesce(col("source_file_name").cast("string"), lit("")),
                coalesce(col("encounter_id").cast("string"), lit("")),
                coalesce(col("patient_id").cast("string"), lit("")),
                coalesce(col("provider_id").cast("string"), lit("")),
                coalesce(col("facility_id").cast("string"), lit("")),
                coalesce(
                    col("encounter_timestamp").cast("string"),
                    lit("")
                ),
                coalesce(col("encounter_type").cast("string"), lit("")),
                coalesce(
                    col("primary_diagnosis_code").cast("string"),
                    lit("")
                ),
                coalesce(col("procedure_code").cast("string"), lit("")),
                coalesce(col("billing_amount").cast("string"), lit("")),
                coalesce(
                    col("insurance_claim_status").cast("string"),
                    lit("")
                ),
                coalesce(col("facility_city").cast("string"), lit("")),
                coalesce(col("facility_country").cast("string"), lit("")),
                coalesce(
                    col("attending_staff_id").cast("string"),
                    lit("")
                ),
                coalesce(col("error_reason").cast("string"), lit(""))
            ),
            256
        )
    )
)


# Preserve repeated identical rejected rows by assigning an occurrence number
quarantine_occurrence_window = (
    Window
    .partitionBy("_quarantine_base_hash")
    .orderBy(lit(1))
)


quarantine_clinical_source_df = (
    quarantine_with_base_hash_df

    .withColumn(
        "_quarantine_occurrence",
        row_number().over(quarantine_occurrence_window)
    )

    .withColumn(
        "quarantine_record_id",
        sha2(
            concat_ws(
                "||",
                col("_quarantine_base_hash"),
                col("_quarantine_occurrence").cast("string")
            ),
            256
        )
    )

    .drop(
        "_quarantine_base_hash",
        "_quarantine_occurrence"
    )
)


expected_quarantine_count = (
    final_quarantine_clinical_encounters_df.count()
)

quarantine_source_count = (
    quarantine_clinical_source_df.count()
)


if expected_quarantine_count != quarantine_source_count:
    raise ValueError(
        f"Quarantine source preparation failed. "
        f"Expected={expected_quarantine_count}, "
        f"Prepared={quarantine_source_count}"
    )


(
    quarantine_clinical_source_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(quarantine_table_name)
)


actual_quarantine_count = (
    spark.table(quarantine_table_name).count()
)


print("Expected quarantine rows:", expected_quarantine_count)
print("Written quarantine rows:", actual_quarantine_count)


if expected_quarantine_count != actual_quarantine_count:
    raise ValueError(
        f"Quarantine write verification failed. "
        f"Expected={expected_quarantine_count}, "
        f"Actual={actual_quarantine_count}"
    )


duplicate_quarantine_ids = (
    spark.table(quarantine_table_name)
    .groupBy("quarantine_record_id")
    .count()
    .filter(col("count") > 1)
    .count()
)


if duplicate_quarantine_ids > 0:
    raise ValueError(
        f"Quarantine contains "
        f"{duplicate_quarantine_ids} duplicate IDs."
    )


print(
    "SUCCESS: Every rejected Bronze row was preserved "
    "in quarantine with a unique record ID."
)

StatementMeta(, 816ca5cb-8499-4444-8822-20e729a2e8b1, 10, Finished, Available, Finished, False)

Expected quarantine rows: 972299
Written quarantine rows: 972299
SUCCESS: Every rejected Bronze row was preserved in quarantine with a unique record ID.


In [9]:
# CELL 8 - UPSERT CLINICAL ENCOUNTERS AUDIT RECORD
# Fully rerunnable: same batch_id + dataset_name updates one audit row

from delta.tables import DeltaTable
from pyspark.sql.functions import (
    col,
    lit,
    current_timestamp,
    xxhash64,
    pmod
)

audit_table_name = (
    "healthcare_control_lakehouse.dbo.audit_table"
)

dataset_name = "clinical_encounters"


# 1. GET EXACTLY ONE BATCH ID

batch_ids = [
    row["batch_id"]
    for row in (
        bronze_clinical_encounters_df
        .select("batch_id")
        .where(col("batch_id").isNotNull())
        .distinct()
        .collect()
    )
]

if len(batch_ids) != 1:
    raise ValueError(
        f"Expected exactly one batch_id, found: {batch_ids}"
    )

current_batch_id = batch_ids[0]


# 2. CALCULATE CURRENT BATCH COUNTS

rows_received = (
    bronze_clinical_encounters_df
    .filter(col("batch_id") == current_batch_id)
    .count()
)

rows_written = (
    deduplicated_clinical_encounters_df
    .filter(col("batch_id") == current_batch_id)
    .count()
)

rejected_rows = (
    final_quarantine_clinical_encounters_df
    .filter(col("batch_id") == current_batch_id)
    .count()
)


if rows_received != rows_written + rejected_rows:
    raise ValueError(
        f"Audit reconciliation failed. "
        f"Received={rows_received}, "
        f"Written={rows_written}, "
        f"Rejected={rejected_rows}"
    )


audit_status = "COMPLETED"


# 3. CREATE DETERMINISTIC AUDIT ID

audit_id = (
    spark.range(1)
    .select(
        pmod(
            xxhash64(
                lit(current_batch_id),
                lit(dataset_name)
            ),
            lit(9223372036854775806)
        )
        .cast("long")
        .alias("audit_id")
    )
    .first()["audit_id"]
)


audit_source_df = (
    spark.range(1)
    .select(
        lit(audit_id)
        .cast("long")
        .alias("audit_id"),

        lit(current_batch_id)
        .alias("batch_id"),

        lit(dataset_name)
        .alias("dataset_name"),

        current_timestamp()
        .alias("start_timestamp"),

        current_timestamp()
        .alias("end_timestamp"),

        lit(rows_received)
        .cast("long")
        .alias("rows_received"),

        lit(rows_written)
        .cast("long")
        .alias("rows_written"),

        lit(rejected_rows)
        .cast("long")
        .alias("rejected_rows"),

        lit(audit_status)
        .alias("status")
    )
)


# 4. VERIFY AUDIT TABLE

if not spark.catalog.tableExists(audit_table_name):
    raise ValueError(
        f"Audit table does not exist: {audit_table_name}"
    )


audit_columns = spark.table(audit_table_name).columns

required_audit_columns = [
    "batch_id",
    "dataset_name",
    "start_timestamp",
    "end_timestamp",
    "rows_received",
    "rows_written",
    "rejected_rows",
    "status"
]

missing_audit_columns = [
    column_name
    for column_name in required_audit_columns
    if column_name not in audit_columns
]

if missing_audit_columns:
    raise ValueError(
        f"Missing audit-table columns: "
        f"{missing_audit_columns}"
    )


# Support an audit table that does not contain audit_id
if "audit_id" not in audit_columns:
    audit_source_df = audit_source_df.drop("audit_id")


# 5. UPSERT AUDIT RECORD

audit_delta = DeltaTable.forName(
    spark,
    audit_table_name
)


update_values = {
    "start_timestamp": "source.start_timestamp",
    "end_timestamp": "source.end_timestamp",
    "rows_received": "source.rows_received",
    "rows_written": "source.rows_written",
    "rejected_rows": "source.rejected_rows",
    "status": "source.status"
}


insert_values = {
    column_name: f"source.{column_name}"
    for column_name in audit_source_df.columns
}


(
    audit_delta.alias("target")
    .merge(
        audit_source_df.alias("source"),
        """
        target.batch_id = source.batch_id
        AND target.dataset_name = source.dataset_name
        """
    )
    .whenMatchedUpdate(set=update_values)
    .whenNotMatchedInsert(values=insert_values)
    .execute()
)


# 6. VERIFY ONE AUDIT RECORD EXISTS

audit_record_count = (
    spark.table(audit_table_name)
    .filter(
        (col("batch_id") == current_batch_id)
        & (col("dataset_name") == dataset_name)
    )
    .count()
)


if audit_record_count != 1:
    raise ValueError(
        f"Expected exactly one audit record, "
        f"found {audit_record_count}."
    )


print("Audit record merged successfully.")
print("Batch ID:", current_batch_id)
print("Rows received:", rows_received)
print("Rows written:", rows_written)
print("Rejected rows:", rejected_rows)
print("Status:", audit_status)


display(
    spark.table(audit_table_name)
    .filter(
        (col("batch_id") == current_batch_id)
        & (col("dataset_name") == dataset_name)
    )
)

StatementMeta(, 816ca5cb-8499-4444-8822-20e729a2e8b1, 11, Finished, Available, Finished, False)

Audit record merged successfully.
Batch ID: clinical_encounters_batch_001
Rows received: 973589
Rows written: 1290
Rejected rows: 972299
Status: COMPLETED


SynapseWidget(Synapse.DataFrame, b110f417-a1a3-42c1-9136-eb1070d1efd8)

In [10]:
# CELL 9 - UPSERT CLINICAL ENCOUNTERS FILE-TRACKING RECORD
# Fully rerunnable: same file + dataset + batch updates one record

from delta.tables import DeltaTable
from pyspark.sql.functions import (
    col,
    lit,
    current_timestamp,
    xxhash64,
    pmod
)

file_tracking_table_name = (
    "healthcare_control_lakehouse.dbo.file_tracking_table"
)

dataset_name = "clinical_encounters"


# ---------------------------------------------------------
# 1. GET EXACTLY ONE SOURCE FILE
# ---------------------------------------------------------

source_files = [
    row["source_file_name"]
    for row in (
        bronze_clinical_encounters_df
        .select("source_file_name")
        .where(col("source_file_name").isNotNull())
        .distinct()
        .collect()
    )
]

if len(source_files) != 1:
    raise ValueError(
        f"Expected exactly one source file, found: {source_files}"
    )

current_source_file = source_files[0]
processing_status = "COMPLETED"


# ---------------------------------------------------------
# 2. CREATE DETERMINISTIC FILE-TRACKING ID
# ---------------------------------------------------------

file_tracking_id = (
    spark.range(1)
    .select(
        (
            pmod(
                xxhash64(
                    lit(current_source_file),
                    lit(dataset_name),
                    lit(current_batch_id)
                ),
                lit(2147483646)
            ) + lit(1)
        )
        .cast("int")
        .alias("file_tracking_id")
    )
    .first()["file_tracking_id"]
)


file_tracking_source_df = (
    spark.range(1)
    .select(
        lit(file_tracking_id)
        .cast("int")
        .alias("file_tracking_id"),

        lit(current_source_file)
        .alias("source_file_name"),

        lit(dataset_name)
        .alias("dataset_name"),

        lit(current_batch_id)
        .alias("batch_id"),

        current_timestamp()
        .alias("processed_timestamp"),

        lit(processing_status)
        .alias("processing_status")
    )
)


# ---------------------------------------------------------
# 3. VERIFY FILE-TRACKING TABLE
# ---------------------------------------------------------

if not spark.catalog.tableExists(file_tracking_table_name):
    raise ValueError(
        f"File-tracking table does not exist: "
        f"{file_tracking_table_name}"
    )


file_tracking_columns = spark.table(
    file_tracking_table_name
).columns


required_file_tracking_columns = [
    "source_file_name",
    "dataset_name",
    "batch_id",
    "processed_timestamp",
    "processing_status"
]


missing_file_tracking_columns = [
    column_name
    for column_name in required_file_tracking_columns
    if column_name not in file_tracking_columns
]


if missing_file_tracking_columns:
    raise ValueError(
        f"Missing file-tracking columns: "
        f"{missing_file_tracking_columns}"
    )


# Support a table without file_tracking_id
if "file_tracking_id" not in file_tracking_columns:
    file_tracking_source_df = (
        file_tracking_source_df.drop("file_tracking_id")
    )


# ---------------------------------------------------------
# 4. UPSERT FILE-TRACKING RECORD
# ---------------------------------------------------------

file_tracking_delta = DeltaTable.forName(
    spark,
    file_tracking_table_name
)


update_values = {
    "processed_timestamp": "source.processed_timestamp",
    "processing_status": "source.processing_status"
}


insert_values = {
    column_name: f"source.{column_name}"
    for column_name in file_tracking_source_df.columns
}


(
    file_tracking_delta.alias("target")
    .merge(
        file_tracking_source_df.alias("source"),
        """
        target.source_file_name = source.source_file_name
        AND target.dataset_name = source.dataset_name
        AND target.batch_id = source.batch_id
        """
    )
    .whenMatchedUpdate(set=update_values)
    .whenNotMatchedInsert(values=insert_values)
    .execute()
)


# ---------------------------------------------------------
# 5. VERIFY ONE FILE-TRACKING RECORD EXISTS
# ---------------------------------------------------------

file_tracking_record_count = (
    spark.table(file_tracking_table_name)
    .filter(
        (col("source_file_name") == current_source_file)
        & (col("dataset_name") == dataset_name)
        & (col("batch_id") == current_batch_id)
    )
    .count()
)


if file_tracking_record_count != 1:
    raise ValueError(
        f"Expected exactly one file-tracking record, "
        f"found {file_tracking_record_count}."
    )


print("File-tracking record merged successfully.")
print("Source file:", current_source_file)
print("Dataset:", dataset_name)
print("Batch ID:", current_batch_id)
print("Processing status:", processing_status)


display(
    spark.table(file_tracking_table_name)
    .filter(
        (col("source_file_name") == current_source_file)
        & (col("dataset_name") == dataset_name)
        & (col("batch_id") == current_batch_id)
    )
)

StatementMeta(, 816ca5cb-8499-4444-8822-20e729a2e8b1, 12, Finished, Available, Finished, False)

File-tracking record merged successfully.
Source file: clinical_encounters.json
Dataset: clinical_encounters
Batch ID: clinical_encounters_batch_001
Processing status: COMPLETED


SynapseWidget(Synapse.DataFrame, 2ce3b3c3-8ac8-408d-8285-cddc4831dc73)

In [11]:
# CELL 10 - UPSERT CLINICAL ENCOUNTERS CONTROL RECORD
# Fully rerunnable: same dataset_name updates one control record

from delta.tables import DeltaTable
from pyspark.sql.functions import (
    col,
    lit,
    current_timestamp,
    xxhash64,
    pmod
)

control_table_name = (
    "healthcare_control_lakehouse.dbo.control_table"
)

dataset_name = "clinical_encounters"
source_table_name = "bronze_clinical_encounters"
target_table_name = "silver_clinical_encounters"
quarantine_table_name = "quarantine_clinical_encounters"
processing_status = "COMPLETED"


# ---------------------------------------------------------
# 1. VERIFY CONTROL TABLE EXISTS
# ---------------------------------------------------------

if not spark.catalog.tableExists(control_table_name):
    raise ValueError(
        f"Control table does not exist: "
        f"{control_table_name}"
    )


control_columns = spark.table(
    control_table_name
).columns


if "dataset_name" not in control_columns:
    raise ValueError(
        "control_table must contain dataset_name."
    )


# ---------------------------------------------------------
# 2. CREATE DETERMINISTIC CONTROL ID
# ---------------------------------------------------------

control_id = (
    spark.range(1)
    .select(
        (
            pmod(
                xxhash64(
                    lit(dataset_name)
                ),
                lit(2147483646)
            ) + lit(1)
        )
        .cast("int")
        .alias("control_id")
    )
    .first()["control_id"]
)


# ---------------------------------------------------------
# 3. PREPARE CONTROL SOURCE RECORD
# ---------------------------------------------------------

control_source_df = (
    spark.range(1)
    .select(
        lit(control_id)
        .cast("int")
        .alias("control_id"),

        lit(dataset_name)
        .alias("dataset_name"),

        lit(current_source_file)
        .alias("source_file_name"),

        lit(source_table_name)
        .alias("source_table_name"),

        lit(target_table_name)
        .alias("target_table_name"),

        lit(quarantine_table_name)
        .alias("quarantine_table_name"),

        lit(current_batch_id)
        .alias("last_processed_batch_id"),

        current_timestamp()
        .alias("last_processed_timestamp"),

        lit(processing_status)
        .alias("processing_status"),

        lit(True)
        .cast("boolean")
        .alias("is_active")
    )
)


# Keep only columns that exist in the current control table
available_control_columns = [
    column_name
    for column_name in control_source_df.columns
    if column_name in control_columns
]


control_source_df = (
    control_source_df
    .select(*available_control_columns)
)


# ---------------------------------------------------------
# 4. UPSERT CONTROL RECORD
# ---------------------------------------------------------

control_delta = DeltaTable.forName(
    spark,
    control_table_name
)


update_values = {
    column_name: f"source.{column_name}"
    for column_name in control_source_df.columns
    if column_name not in [
        "control_id",
        "dataset_name"
    ]
}


insert_values = {
    column_name: f"source.{column_name}"
    for column_name in control_source_df.columns
}


control_merge = (
    control_delta.alias("target")
    .merge(
        control_source_df.alias("source"),
        """
        target.dataset_name = source.dataset_name
        """
    )
)


if update_values:
    control_merge = (
        control_merge
        .whenMatchedUpdate(
            set=update_values
        )
    )


(
    control_merge
    .whenNotMatchedInsert(
        values=insert_values
    )
    .execute()
)


# ---------------------------------------------------------
# 5. VERIFY ONE CONTROL RECORD EXISTS
# ---------------------------------------------------------

control_record_count = (
    spark.table(control_table_name)
    .filter(
        col("dataset_name") == dataset_name
    )
    .count()
)


if control_record_count != 1:
    raise ValueError(
        f"Expected exactly one control record, "
        f"found {control_record_count}."
    )


print("Control record merged successfully.")
print("Control ID:", control_id)
print("Dataset:", dataset_name)
print("Batch ID:", current_batch_id)
print("Processing status:", processing_status)


display(
    spark.table(control_table_name)
    .filter(
        col("dataset_name") == dataset_name
    )
)

StatementMeta(, 816ca5cb-8499-4444-8822-20e729a2e8b1, 13, Finished, Available, Finished, False)

Control record merged successfully.
Control ID: 100574598
Dataset: clinical_encounters
Batch ID: clinical_encounters_batch_001
Processing status: COMPLETED


SynapseWidget(Synapse.DataFrame, 0cb72527-04e5-4e51-8948-2fb3f0cf7fd0)

In [12]:
# CELL 11 - FINAL SILVER DATA-QUALITY VERIFICATION
# Fully rerunnable: stops the pipeline if dirty or duplicate
# records remain in the Silver table

from pyspark.sql.functions import (
    col,
    current_timestamp
)

silver_verification_df = spark.table(
    "silver_clinical_encounters"
)


# ---------------------------------------------------------
# 1. CHECK DUPLICATE ENCOUNTER IDS
# ---------------------------------------------------------

duplicate_encounter_count = (
    silver_verification_df
    .groupBy("encounter_id")
    .count()
    .filter(col("count") > 1)
    .count()
)


# ---------------------------------------------------------
# 2. CHECK INVALID IDENTIFIERS
# ---------------------------------------------------------

invalid_identifier_count = (
    silver_verification_df
    .filter(
        col("encounter_id").isNull()
        | ~col("encounter_id").rlike(
            r"^ENC_[0-9]{7}$"
        )
        | col("patient_id").isNull()
        | ~col("patient_id").rlike(
            r"^PAT_[0-9]{6}$"
        )
        | col("provider_id").isNull()
        | ~col("provider_id").rlike(
            r"^PROV_[0-9]{4}$"
        )
        | col("facility_id").isNull()
        | ~col("facility_id").rlike(
            r"^FAC_[0-9]+$"
        )
        | col("attending_staff_id").isNull()
        | ~col("attending_staff_id").rlike(
            r"^EMP_[0-9]{4}$"
        )
    )
    .count()
)


# ---------------------------------------------------------
# 3. CHECK INVALID TIMESTAMPS
# ---------------------------------------------------------

invalid_timestamp_count = (
    silver_verification_df
    .filter(
        col("encounter_timestamp").isNull()
        | (
            col("encounter_timestamp")
            < "2000-01-01 00:00:00"
        )
        | (
            col("encounter_timestamp")
            > current_timestamp()
        )
    )
    .count()
)


# ---------------------------------------------------------
# 4. CHECK INVALID CATEGORIES
# ---------------------------------------------------------

invalid_category_count = (
    silver_verification_df
    .filter(
        col("encounter_type").isNull()
        | ~col("encounter_type").isin(
            "INPATIENT",
            "OUTPATIENT",
            "EMERGENCY"
        )
        | col("insurance_claim_status").isNull()
        | ~col("insurance_claim_status").isin(
            "APPROVED",
            "PENDING",
            "PAID",
            "DENIED"
        )
        | col("facility_country").isNull()
        | (col("facility_country") != "CANADA")
    )
    .count()
)


# ---------------------------------------------------------
# 5. CHECK INVALID BILLING AMOUNTS
# ---------------------------------------------------------

invalid_billing_count = (
    silver_verification_df
    .filter(
        col("billing_amount").isNull()
        | (col("billing_amount") < 0)
        | (col("billing_amount") > 1000000)
    )
    .count()
)


# ---------------------------------------------------------
# 6. CHECK INVALID CODES AND CITY VALUES
# ---------------------------------------------------------

invalid_code_or_city_count = (
    silver_verification_df
    .filter(
        col("primary_diagnosis_code").isNull()
        | ~col("primary_diagnosis_code").rlike(
            r"^[A-Z][0-9]{2}(\.[0-9A-Z]+)?$"
        )
        | col("procedure_code").isNull()
        | ~col("procedure_code").rlike(
            r"^[0-9]{5}$"
        )
        | col("facility_city").isNull()
        | ~col("facility_city").rlike(
            r"^[A-Za-z][A-Za-z .'-]*$"
        )
    )
    .count()
)


# ---------------------------------------------------------
# 7. CHECK REQUIRED HASH AND METADATA
# ---------------------------------------------------------

invalid_metadata_count = (
    silver_verification_df
    .filter(
        col("batch_id").isNull()
        | col("source_file_name").isNull()
        | col("row_hash").isNull()
        | col("silver_processing_timestamp").isNull()
    )
    .count()
)


print(
    "Duplicate encounter IDs:",
    duplicate_encounter_count
)

print(
    "Invalid identifiers:",
    invalid_identifier_count
)

print(
    "Invalid timestamps:",
    invalid_timestamp_count
)

print(
    "Invalid categories:",
    invalid_category_count
)

print(
    "Invalid billing amounts:",
    invalid_billing_count
)

print(
    "Invalid codes or cities:",
    invalid_code_or_city_count
)

print(
    "Invalid metadata:",
    invalid_metadata_count
)


total_quality_failures = (
    duplicate_encounter_count
    + invalid_identifier_count
    + invalid_timestamp_count
    + invalid_category_count
    + invalid_billing_count
    + invalid_code_or_city_count
    + invalid_metadata_count
)


if total_quality_failures > 0:

    raise ValueError(
        f"Silver quality verification failed. "
        f"Total detected failures="
        f"{total_quality_failures}"
    )


print(
    "SUCCESS: Silver clinical encounters contains "
    "no duplicate, invalid or untracked records."
)


display(
    silver_verification_df
    .select(
        "encounter_id",
        "patient_id",
        "facility_id",
        "encounter_timestamp",
        "encounter_type",
        "billing_amount",
        "insurance_claim_status",
        "facility_city",
        "facility_country",
        "batch_id",
        "row_hash"
    )
    .orderBy("encounter_id")
    .limit(20)
)

StatementMeta(, 816ca5cb-8499-4444-8822-20e729a2e8b1, 14, Finished, Available, Finished, False)

Duplicate encounter IDs: 0
Invalid identifiers: 0
Invalid timestamps: 0
Invalid categories: 0
Invalid billing amounts: 0
Invalid codes or cities: 0
Invalid metadata: 0
SUCCESS: Silver clinical encounters contains no duplicate, invalid or untracked records.


SynapseWidget(Synapse.DataFrame, 8a14bbaf-86c9-45b4-8aae-12405633d68a)

In [13]:
# CELL 12 - FINAL PIPELINE RECONCILIATION AND COMPLETION
# Fully rerunnable: confirms Bronze = Silver + Quarantine
# and verifies audit, file tracking and control records

from pyspark.sql.functions import col


bronze_final_count = (
    spark.table("bronze_clinical_encounters").count()
)

silver_final_count = (
    spark.table("silver_clinical_encounters").count()
)

quarantine_final_count = (
    spark.table("quarantine_clinical_encounters").count()
)


reconciled_final_count = (
    silver_final_count
    + quarantine_final_count
)


print("Bronze rows:", bronze_final_count)
print("Silver rows:", silver_final_count)
print("Quarantine rows:", quarantine_final_count)
print("Silver + Quarantine:", reconciled_final_count)


if bronze_final_count != reconciled_final_count:

    raise ValueError(
        f"Final reconciliation failed. "
        f"Bronze={bronze_final_count}, "
        f"Silver+Quarantine={reconciled_final_count}"
    )


# ---------------------------------------------------------
# VERIFY AUDIT RECORD
# ---------------------------------------------------------

audit_verification_df = (
    spark.table(
        "healthcare_control_lakehouse.dbo.audit_table"
    )
    .filter(
        (col("batch_id") == current_batch_id)
        & (col("dataset_name") == dataset_name)
    )
)


audit_record_count = audit_verification_df.count()


if audit_record_count != 1:

    raise ValueError(
        f"Audit verification failed. "
        f"Expected 1 record, found {audit_record_count}."
    )


audit_record = audit_verification_df.first()


if (
    audit_record["rows_received"] != bronze_final_count
    or audit_record["rows_written"] != silver_final_count
    or audit_record["rejected_rows"] != quarantine_final_count
    or audit_record["status"] != "COMPLETED"
):

    raise ValueError(
        "Audit values do not match the final table counts."
    )


# ---------------------------------------------------------
# VERIFY FILE-TRACKING RECORD
# ---------------------------------------------------------

file_tracking_verification_count = (
    spark.table(
        "healthcare_control_lakehouse.dbo.file_tracking_table"
    )
    .filter(
        (col("source_file_name") == current_source_file)
        & (col("dataset_name") == dataset_name)
        & (col("batch_id") == current_batch_id)
        & (col("processing_status") == "COMPLETED")
    )
    .count()
)


if file_tracking_verification_count != 1:

    raise ValueError(
        f"File-tracking verification failed. "
        f"Expected 1 completed record, "
        f"found {file_tracking_verification_count}."
    )


# ---------------------------------------------------------
# VERIFY CONTROL RECORD
# ---------------------------------------------------------

control_verification_count = (
    spark.table(
        "healthcare_control_lakehouse.dbo.control_table"
    )
    .filter(
        col("dataset_name") == dataset_name
    )
    .count()
)


if control_verification_count != 1:

    raise ValueError(
        f"Control verification failed. "
        f"Expected 1 record, "
        f"found {control_verification_count}."
    )


# ---------------------------------------------------------
# FINAL PIPELINE RESULT
# ---------------------------------------------------------

rejection_percentage = (
    round(
        (
            quarantine_final_count
            / bronze_final_count
        ) * 100,
        2
    )
    if bronze_final_count > 0
    else 0
)


print("")
print("==========================================")
print("CLINICAL ENCOUNTERS SILVER PIPELINE PASSED")
print("==========================================")
print("Dataset:", dataset_name)
print("Batch ID:", current_batch_id)
print("Source file:", current_source_file)
print("Bronze rows:", bronze_final_count)
print("Silver rows:", silver_final_count)
print("Quarantine rows:", quarantine_final_count)
print("Rejected percentage:", rejection_percentage, "%")
print("Audit status: COMPLETED")
print("File tracking status: COMPLETED")
print("Control record: VERIFIED")
print("Final reconciliation: PASSED")
print("==========================================")


display(
    spark.createDataFrame(
        [
            (
                dataset_name,
                current_batch_id,
                current_source_file,
                bronze_final_count,
                silver_final_count,
                quarantine_final_count,
                rejection_percentage,
                "COMPLETED"
            )
        ],
        [
            "dataset_name",
            "batch_id",
            "source_file_name",
            "bronze_rows",
            "silver_rows",
            "quarantine_rows",
            "rejection_percentage",
            "pipeline_status"
        ]
    )
)

# Release cached memory after the complete notebook finishes
silver_clinical_source_df.unpersist()

spark.catalog.clearCache()

print("SUCCESS: Spark cache released.")

StatementMeta(, 816ca5cb-8499-4444-8822-20e729a2e8b1, 15, Finished, Available, Finished, False)

Bronze rows: 973589
Silver rows: 1290
Quarantine rows: 972299
Silver + Quarantine: 973589

CLINICAL ENCOUNTERS SILVER PIPELINE PASSED
Dataset: clinical_encounters
Batch ID: clinical_encounters_batch_001
Source file: clinical_encounters.json
Bronze rows: 973589
Silver rows: 1290
Quarantine rows: 972299
Rejected percentage: 99.87 %
Audit status: COMPLETED
File tracking status: COMPLETED
Control record: VERIFIED
Final reconciliation: PASSED


SynapseWidget(Synapse.DataFrame, 0b33edf3-a111-407c-aa08-4e509dfae282)

SUCCESS: Spark cache released.
